# Importing Libraries

In [14]:
import warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [25]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [16]:
# 1. Chargement du dataset nettoyé (74 111 lignes - Sans valeurs manquantes)
df_cleaned = pd.read_csv("data/cleaned_dataset.csv")

In [23]:
X.head()

,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,instant_bookable,latitude,longitude,number_of_reviews
0,Apartment,Entire home/apt,152,3,1.0,Real Bed,strict,True,NYC,0,40.696524,-73.991617,2
1,Apartment,Entire home/apt,218,7,1.0,Real Bed,strict,True,NYC,1,40.766115,-73.989040,6
2,Apartment,Entire home/apt,311,5,1.0,Real Bed,moderate,True,NYC,1,40.808110,-73.943756,10
3,House,Entire home/apt,210,4,1.0,Real Bed,flexible,True,SF,0,37.772004,-122.431619,0
4,Apartment,Entire home/apt,174,2,1.0,Real Bed,moderate,True,DC,1,38.925627,-77.034596,4


In [17]:
# 2. Séparation en caractéristiques (X) et cible (y)
X = df_cleaned.drop(["log_price"], axis=1)
y = df_cleaned["log_price"]

In [18]:
# 3. Découpage standard (Robuste grâce aux 74k lignes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
# 4. Identification des colonnes pour le Preprocessing Pipeline
num_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object"]).columns.tolist()

In [20]:
# Pipeline de transformation avec RobustScaler pour neutraliser les outliers
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

In [21]:
# 5. Banque complète de modèles de régression
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression (L2)": Ridge(alpha=1.0),
    "Lasso Regression (L1)": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42, n_jobs=-1),
    "LightGBM Regressor": LGBMRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost Regressor": CatBoostRegressor(iterations=150, learning_rate=0.08, depth=6, random_state=42, verbose=0)
}

In [22]:
# 6. Boucle d'entraînement et d'évaluation automatisée
results = []

print(f"Entraînement lancé sur {len(X_train)} lignes propres...")
print("Évaluation en cours sur les 10 modèles de régression...\n")

for name, model in models.items():
    model_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("regressor", model)]
    )
    
    # Entraînement direct
    model_pipeline.fit(X_train, y_train)
    
    # Prédictions
    y_pred = model_pipeline.predict(X_test)
    
    # Calcul des métriques
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    # Calcul du R² Ajusté
    n = len(y_test)
    X_test_transformed = preprocessor.transform(X_test)
    p = X_test_transformed.shape[1]
    adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))
    
    results.append({
        "Model": name, 
        "R² Score": r2, 
        "Adj R² Score": adj_r2,
        "RMSE": rmse, 
        "MAE": mae
    })

# 7. Synthèse comparative finale ordonnée par le R² Ajusté
df_results = pd.DataFrame(results).sort_values(by="Adj R² Score", ascending=False)
print(df_results.to_string(index=False))

Entraînement lancé sur 59288 lignes propres...
Évaluation en cours sur les 10 modèles de régression...

                Model  R² Score  Adj R² Score     RMSE      MAE
    XGBoost Regressor  0.696605      0.695372 0.394793 0.289072
   LightGBM Regressor  0.693619      0.692373 0.396732 0.290696
    Gradient Boosting  0.686876      0.685603 0.401074 0.294152
        Random Forest  0.684394      0.683111 0.402660 0.294489
   CatBoost Regressor  0.680518      0.679219 0.405125 0.297590
        Decision Tree  0.634420      0.632934 0.433368 0.315949
    Linear Regression  0.566938      0.565177 0.471673 0.354892
Ridge Regression (L2)  0.552816      0.550998 0.479302 0.362605
           ElasticNet  0.531850      0.529948 0.490409 0.369320
Lasso Regression (L1)  0.520564      0.518615 0.496285 0.373322


In [ ]:
# 5. Define the Hyperparameter Tuning Grid for XGBoost
# We focus on parameters that control overfitting and learning speed
param_distributions = {
    'model__n_estimators': [100, 300, 500, 800],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [3, 5, 7, 9],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__reg_alpha': [0, 0.1, 1, 10],      # L1 regularization
    'model__reg_lambda': [1, 2, 5, 10]        # L2 regularization
}

# 6. Create the base pipeline with a placeholder XGBoost model
base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, n_jobs=-1))
])

# 7. Set up the Randomized Search
print("--- Starting Hyperparameter Fine-Tuning ---")
random_search = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_distributions,
    n_iter=20,                 # Evaluates 20 random combinations of parameters
    cv=3,                      # 3-fold cross-validation
    scoring='r2',              # Optimize for R2 score
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# Run the search on your training data
random_search.fit(X_train, y_train)

# 8. Extract the absolute best model pipeline
best_pipeline = random_search.best_estimator_

print("\nBest Hyperparameters Found:")
print(random_search.best_params_)

# 9. Evaluate the Final Fine-Tuned Model on the Test Set
y_pred = best_pipeline.predict(X_test)

final_r2 = r2_score(y_test, y_pred)
final_mae = mean_absolute_error(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- Final Fine-Tuned XGBoost Performance ---")
print(f"Final R² Score: {final_r2:.6f}")
print(f"Final RMSE:     {final_rmse:.6f}")
print(f"Final MAE:      {final_mae:.6f}")

# 10. Export and save the model for future production/inference use
joblib.dump(best_pipeline, "final_tuned_xgboost_model.pkl")
print("\nSuccess: 'final_tuned_xgboost_model.pkl' has been saved and is ready for production!")

--- Starting Hyperparameter Fine-Tuning ---
Fitting 3 folds for each of 20 candidates, totalling 60 fits
